# 🔁 25차원 재학습 Colab 재현 — 동일 v2 파라미터 · 차원 vs 튜닝 분리

> **목적**: 로컬(lightgbm 4.7.0) 실행 산출물 `benchmarks/2026-10-14/tuned_25d/` 의 `best_iteration`·지표를 **Colab 에서 재현**하고, `models/` 교체용 공식 산출물을 확보합니다.
> **원본 실험**: `benchmarks/retrain_25d.py` (스키마 SSoT 의 `metrics.*.tuned.best_params` 고정 + 25차원 서빙 서브셋)

## 이 노트북만 업로드하면 됩니다
`.py` 스크립트·모듈 import 가 **필요 없습니다.** 레포의 `benchmarks/CastTransit.ipynb` 에서 전처리 셀 4개(`[Cell 3]`/`[Cell 4-A]`/`[Cell 4-B]`/`[Cell 5]`)만 **그대로 exec** 하므로 피처 드리프트가 0입니다.

## 📥 사전 준비 — Drive 에 레포 · 원천 CSV 두기
```bash
# 1) 레포가 Drive 에 없을 때 (이 노트북은 '커밋된' CastTransit.ipynb 만 읽음 → clone 으로 충분)
!git clone https://github.com/14EE4/weather-transit-web.git /content/drive/MyDrive/weather-transit-web
```
2) 원천 CSV `data/2025_서울25개구_교통날씨_전처리_2.csv`(약 28MB)를 같은 경로에 업로드합니다 — `.gitignore` 의 `*.csv` 로 **레포에 포함되지 않으므로 clone 만으로는 생기지 않습니다.**
> `retrain_25d.py`·`feature_schema.py`(로컬 미푸시)를 import 하지 않으므로 **구버전 clone 으로도 재현됩니다.**

## 실행 순서
1. `[Step 0]` 환경 준비(패키지 + 버전 기록)
2. `[Step 1]` 드라이브 마운트 + 레포/CSV 확인
3. `[Step 2]` 원본 노트북 전처리 셀 4개 실행 — **[Cell 6]/[Cell 6-1] Optuna 는 건너뜁니다**
4. `[Step 3]` 25차원 서브셋 재학습(파라미터 고정)
5. `[Step 4]` ONNX 3종 + 25차원 스키마 반출
6. `[Step 5]` 결과 표 확인(로컬값·v2 32d 참고치와 비교)

## ⚠️ 주의
- 사전 조건: Drive 에 `weather-transit-web` 레포와 `data/2025_서울25개구_교통날씨_전처리_2.csv` 가 있어야 합니다(위 사전 준비 · `CastTransit.ipynb` [Cell 3] 과 동일 경로).
- 반출 폴더는 `benchmarks/2026-10-14/tuned_25d_colab/` 로 고정 → **로컬 실행본 `tuned_25d/` 를 덮어쓰지 않습니다.**
- `best_iteration` 완전 일치는 요구하지 않습니다(스레드·플랫폼 차이). **지표가 소수 셋째 자리 내면 재현 성공**입니다.

In [ ]:
# [Step 0] Colab 환경 준비 — 로컬(lightgbm 4.7.0)과의 차이를 먼저 '기록'합니다.
#  - 코드 재현성 확인이 목적이므로 로컬과 같은 버전을 핀합니다.
#  - 환경(버전) 효과를 보려면 아래 핀(`lightgbm==4.7.0`)을 제거하고 1회 더 실행해 별도 폴더로 반입하세요.
!pip install -q lightgbm==4.7.0 onnxmltools skl2onnx onnxruntime

import sys, platform
import numpy, pandas, lightgbm

print("python   :", sys.version.split()[0], "|", platform.platform())
print("lightgbm :", lightgbm.__version__, "| numpy:", numpy.__version__, "| pandas:", pandas.__version__)
print()
print("⚠️ lightgbm 이 4.7.0 이 아니면 [런타임 → 세션 다시 시작] 후 [Step 1] 부터 다시 실행하세요.")

In [ ]:
# [Step 1] 드라이브 마운트 + 레포/원천 데이터 존재 확인
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

REPO = Path('/content/drive/MyDrive/weather-transit-web')   # CastTransit.ipynb [Cell 3] 의 CSV 경로와 동일 레포
SRC_NB = REPO / 'benchmarks' / 'CastTransit.ipynb'
CSV_PATH = REPO / 'data' / '2025_서울25개구_교통날씨_전처리_2.csv'
assert SRC_NB.is_file(), f"원본 노트북을 찾지 못했습니다: {SRC_NB}"
assert CSV_PATH.is_file(), f"원천 CSV 를 찾지 못했습니다: {CSV_PATH}"

print("REPO     :", REPO)
print("notebook :", SRC_NB.name)
print("CSV      :", CSV_PATH.name, f"({CSV_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# [Step 2] 원본 노트북의 전처리 셀 4개를 '그대로' 실행합니다(복사·재작성 없음 → 피처 드리프트 0).
#   실행: [Cell 3] 로드·결측 정제 · [Cell 4-A] 캘린더8+기상8+공간5 · [Cell 4-B] 시계열 래그 11 → 32차원 · [Cell 5] 분할
#   건너뜀: [Cell 6] 베이스라인 · [Cell 6-1] Optuna(30 trials) · [Cell 7] 32차원 export
#     → Optuna 는 seed 가 없어 재탐색 시 파라미터가 달라져(비교 가능성 훼손) 20분 이상 소요됩니다.
_nb = json.loads(SRC_NB.read_text(encoding='utf-8'))


def cell_source(tag: str) -> str:
    """원본 노트북에서 첫 줄 태그(`# [Cell 3]` 등)로 코드 셀을 찾습니다(인덱스 하드코딩 금지)."""
    for c in _nb['cells']:
        if c['cell_type'] == 'code' and ''.join(c['source']).lstrip().startswith(tag):
            return ''.join(c['source'])
    raise RuntimeError(f"원본 노트북에서 '{tag}' 셀을 찾지 못했습니다(tag 형식 확인)")


for _tag in ('# [Cell 3]', '# [Cell 4-A]', '# [Cell 4-B]', '# [Cell 5]'):
    print(f"\n===================== exec {_tag} =====================")
    exec(compile(cell_source(_tag), f'<CastTransit {_tag}>', 'exec'), globals())

_test_dates = df.loc[test_mask, '기준_날짜']
print(f"\n[ready] df={df.shape} | X_train={X_train.shape} | X_val={X_val.shape} | X_test={X_test.shape}")
print(f"[ready] 피처 {len(FINAL_FEATURE_COLS)}차원(8/8/5/11) | test {_test_dates.min().date()}~{_test_dates.max().date()} (n={len(X_test):,})")

## [Step 3] 직전 메모 — 왜 Optuna 를 다시 돌리지 않는가

원본 노트북 `[Cell 6-1]` 의 `optuna.create_study(direction='minimize')` 에는 **seed 가 없습니다.** 즉 재실행하면 `best_params` 가 달라지고, 그러면 이 실험이 분리하려는 대상(① 차원)이 아니라 ② 튜닝 효과가 다시 섞입니다. 그래서 v2 스키마 SSoT 에 **동결된 `best_params` 를 그대로 주입**합니다(로컬 `retrain_25d.py` 와 동일 조건, Optuna 20분도 절약).

In [ ]:
# [Step 3] 25차원 서빙 서브셋 재학습 — v2 best_params 고정 (차원 vs 튜닝 분리)
#   고정 파라미터 출처: benchmarks/2026-10-14/feature_schema.json :: metrics.<mode>.tuned.best_params
#   25차원 배치: main.py::prepare_session_input 과 동일
#     = 32차원에서 [17..20](공간 인프라 4) + [27..29](same_day_last_week 3) 제외
import numpy as np
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

FROZEN_PARAMS = {
    'bike': {'learning_rate': 0.08811950498925582, 'num_leaves': 109, 'max_depth': 10,
             'subsample': 0.6523792755919405, 'colsample_bytree': 0.6790616323310082,
             'reg_alpha': 0.0024082899040478675, 'reg_lambda': 0.011453737805966376},
    'bus': {'learning_rate': 0.09703869938657278, 'num_leaves': 126, 'max_depth': 10,
            'subsample': 0.8605477792677568, 'colsample_bytree': 0.6661090758204229,
            'reg_alpha': 3.2075582423454496, 'reg_lambda': 0.40019714909533227},
    'subway': {'learning_rate': 0.09807115232677817, 'num_leaves': 117, 'max_depth': 8,
               'subsample': 0.6620109779156034, 'colsample_bytree': 0.7180910290353443,
               'reg_alpha': 0.9148471286354888, 'reg_lambda': 0.085461631351953},
}
TARGETS = {'bike': '따릉이_대여', 'bus': '버스_승차', 'subway': '지하철_승차'}
TRAIN_FRAME = {'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
               'n_estimators': 1500, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
EARLY_STOPPING_ROUNDS = 50

LEGACY_25D_INDICES = list(range(17)) + [21, 22, 23, 24, 25, 26, 30, 31]   # feature_schema.LEGACY_25D_INDICES
COLS_25 = [FINAL_FEATURE_COLS[i] for i in LEGACY_25D_INDICES]
EXCLUDED_7 = [c for c in FINAL_FEATURE_COLS if c not in COLS_25]
assert len(COLS_25) == 25, len(COLS_25)
assert EXCLUDED_7 == ['district_subway_stations', 'district_bus_stops', 'district_bike_racks',
                      'commercial_density', 'same_day_last_week_bike', 'same_day_last_week_bus',
                      'same_day_last_week_sub'], EXCLUDED_7
print("[배치] 25차원:", COLS_25)
print("[배치] 제외 7:", EXCLUDED_7)

Xtr25 = X_train[COLS_25].to_numpy(dtype=np.float32)
Xva25 = X_val[COLS_25].to_numpy(dtype=np.float32)
Xte25 = X_test[COLS_25].to_numpy(dtype=np.float32)

tuned25_models, tuned25_metrics = {}, {}
for mode, target_col in TARGETS.items():
    y_tr = df.loc[train_mask, target_col].astype(np.float64).to_numpy()
    y_va = df.loc[val_mask, target_col].astype(np.float64).to_numpy()
    y_te = df.loc[test_mask, target_col].astype(np.float64).to_numpy()

    params = dict(FROZEN_PARAMS[mode])
    params.update(TRAIN_FRAME)
    model = lgb.LGBMRegressor(**params)
    model.fit(Xtr25, y_tr, eval_set=[(Xva25, y_va)],
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])

    pred = np.clip(model.predict(Xte25), 0.0, None)
    nz = y_te != 0   # MAPE 는 실측 0(심야 운휴 등) 제외 — evaluate_models.regression_metrics 와 동일 규칙
    tuned25_metrics[mode] = {
        'r2': float(r2_score(y_te, pred)),
        'rmse': float(mean_squared_error(y_te, pred) ** 0.5),
        'mae': float(mean_absolute_error(y_te, pred)),
        'mape_pct_nonzero': float(np.mean(np.abs((y_te[nz] - pred[nz]) / y_te[nz])) * 100.0),
        'best_iteration': int(getattr(model, 'best_iteration_', 0) or TRAIN_FRAME['n_estimators']),
        'best_params': dict(FROZEN_PARAMS[mode]),
        'input_dim': 25,
        'n': int(y_te.size),
    }
    tuned25_models[mode] = model
    t = tuned25_metrics[mode]
    print(f"[25d {mode:6s}] R2={t['r2']:.4f} RMSE={t['rmse']:9.3f} MAE={t['mae']:8.3f} "
          f"MAPE(!=0)={t['mape_pct_nonzero']:.2f}% best_iter={t['best_iteration']}")

In [ ]:
# [Step 4] 25차원 ONNX 3종 + 25차원 서빙 스키마 반출 (입력 [None, 25])
#   반출 위치 = 레포의 benchmarks/2026-10-14/tuned_25d_colab/
#     → 로컬 실행본 benchmarks/2026-10-14/tuned_25d/ 를 절대 덮어쓰지 않습니다(비교 대상 보존).
#   구조는 benchmarks/retrain_25d.py::export_assets 와 동일 — 반입 후 정규 하네스가 그대로 읽습니다.
import json
import onnxmltools
import onnxruntime as ort
from onnxmltools.convert.common.data_types import FloatTensorType

OUT_DIR = REPO / 'benchmarks' / '2026-10-14' / 'tuned_25d_colab'
assert OUT_DIR.name == 'tuned_25d_colab', OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
initial_type = [('float_input', FloatTensorType([None, 25]))]

_tr = df.loc[train_mask, '기준_날짜']
_va = df.loc[val_mask, '기준_날짜']
_te = df.loc[test_mask, '기준_날짜']
SPLIT = {'train': f"2025-01-01~{_tr.max().date()}",
         'valid': f"{_va.min().date()}~{_va.max().date()}",
         'test': f"{_te.min().date()}~{_te.max().date()}"}
print('[split]', SPLIT)

schema25 = {
    'model_version': 'v2-전처리_2-25d-retrain-colab',
    'feature_dim': 25,
    'feature_cols': COLS_25,
    'layout': '25d-subset',
    'order': ['calendar(8)', 'weather(8)', 'spatial(1)', 'timeseries(6+2)'],
    'derived_from': {
        'model_version': 'v2-전처리_2-32d',
        'feature_dim_32d': len(FINAL_FEATURE_COLS),
        'note': 'v2 32차원 SSoT 의 25차원 서빙 서브셋 — main.py::prepare_session_input 과 순서 동일',
    },
    'targets': TARGETS,
    'split': SPLIT,
    'split_source': SPLIT_SOURCE,
    'holiday_calendar': 'KR_HOLIDAYS_2025 (19일, main.py::KR_HOLIDAYS 와 동일)',
    'env': {
        'runtime': 'Google Colab',
        'python': sys.version.split()[0],
        'lightgbm': lgb.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
    },
    'metrics': {
        mode: {
            'tuned': {k: tuned25_metrics[mode][k]
                      for k in ('r2', 'rmse', 'mae', 'mape_pct_nonzero', 'best_iteration')},
            'best_params': tuned25_metrics[mode]['best_params'],
        }
        for mode in TARGETS
    },
}

paths = []
for mode, model in tuned25_models.items():
    p = OUT_DIR / f'lgb_{mode}_demand.onnx'
    onnx_model = onnxmltools.convert_lightgbm(model, initial_types=initial_type, target_opset=14)
    p.write_bytes(onnx_model.SerializeToString())
    dim = int(ort.InferenceSession(str(p), providers=['CPUExecutionProvider']).get_inputs()[0].shape[1])
    assert dim == 25, f'ONNX 입력 차원 오류: {p.name} = {dim}'
    paths.append(p)
    print(f"[export] {p.name:24s} {p.stat().st_size:>10,} byte | 입력 {dim}차원")

schema_path = OUT_DIR / 'feature_schema.json'
schema_path.write_text(json.dumps(schema25, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"[export] {schema_path.name} (25차원 서빙 스키마)")
print('\n[반출 완료]', OUT_DIR.as_posix())

try:  # 브라우저 다운로드(Drive 동기화를 쓰면 없어도 됩니다)
    from google.colab import files
    for _p in paths + [schema_path]:
        files.download(str(_p))
except Exception as exc:  # noqa: BLE001
    print('[skip] 브라우저 다운로드 생략:', exc)

In [ ]:
# [Step 5] 대조군 표 — 로컬 tuned_25d(기준값) 및 v2 32차원 참고치와 비교
#   로컬 기준값 출처: benchmarks/2026-10-14/tuned_25d/ (lightgbm 4.7.0, Windows)
import pandas as pd

LOCAL_25D = {   # 출처: benchmarks/results/retrain_25d_20261014_192001.json (직접 모델 지표)
    'bike': {'r2': 0.9883, 'rmse': 16.6664, 'mae': 9.6767, 'best_iteration': 680},
    'bus': {'r2': 0.9912, 'rmse': 528.6187, 'mae': 342.7211, 'best_iteration': 1498},
    'subway': {'r2': 0.9935, 'rmse': 769.3175, 'mae': 330.8713, 'best_iteration': 446},
}
V2_32D = {   # 출처: benchmarks/2026-10-14/feature_schema.json :: metrics.<mode>.tuned (v2 32차원)
    'bike': {'r2': 0.9873, 'rmse': 17.382, 'mae': 9.6694, 'best_iteration': 834},
    'bus': {'r2': 0.9901, 'rmse': 561.352, 'mae': 370.441, 'best_iteration': 1390},
    'subway': {'r2': 0.9951, 'rmse': 662.095, 'mae': 277.6328, 'best_iteration': 865},
}

rows = []
for mode in TARGETS:
    c, l, v = tuned25_metrics[mode], LOCAL_25D[mode], V2_32D[mode]
    rows.append({
        '수단': mode,
        'Colab R²': f"{c['r2']:.4f}",
        'RMSE': f"{c['rmse']:.3f}",
        'MAE': f"{c['mae']:.3f}",
        'MAPE(≠0)': f"{c['mape_pct_nonzero']:.2f}%",
        'best_iter': c['best_iteration'],
        'ΔR²(local)': f"{c['r2'] - l['r2']:+.4f}",
        'Δbest_iter(local)': f"{c['best_iteration'] - l['best_iteration']:+d}",
        'ΔR²(v2 32d)': f"{c['r2'] - v['r2']:+.4f}",
        'ΔMAE(v2 32d)': f"{c['mae'] - v['mae']:+.3f}",
    })

print(pd.DataFrame(rows).to_string(index=False))
print()
print('판정: ΔR²(local) 가 ±0.0005 이내 & Δbest_iter(local) 가 ±10% 이내면 재현 성공(스레드·플랫폼 차이 허용).')

## 검증 이력 (2026-10-14)
- 본 노트북의 `[Step 2]`~`[Step 5]` 셀 로직을 **로컬에서 그대로 실행**해 검증했습니다.
  - 25차원 재학습 지표·`best_iteration` 이 `retrain_25d.py` 실행값과 **완전 일치**(bike 680 / bus 1,498 / subway 446, ΔR² 0.0000).
  - 반출 ONNX 3종은 정규 하네스가 **`dim=25` · `layout=25d-subset`** 으로 자동 판별해 평가(4-way 표 생성) → 반입 후 추가 조치 불필요.
- 즉 **Colab 실행에서 새로 확인할 값은 '플랫폼 편차'뿐**입니다(위 대조 표의 Δ 열).

## 결과 반입(로컬) — 3단계

1. Colab 이 Drive 에 쓴 `benchmarks/2026-10-14/tuned_25d_colab/` 을 로컬 레포로 동기화합니다.
   (브라우저 다운로드를 썼다면 받은 파일 4개를 같은 폴더에 넣으면 됩니다.)
2. 로컬에서 **4-way 비교**(배포본 25d · v1 32d · v2 32d · `tuned_25d_colab` 25d):

   ```powershell
   python benchmarks/evaluate_models.py --model tuned_25d_colab=benchmarks/2026-10-14/tuned_25d_colab
   ```

3. 로컬 실행본까지 포함한 **5-way 비교**:

   ```powershell
   python benchmarks/evaluate_models.py --model tuned_25d_colab=benchmarks/2026-10-14/tuned_25d_colab --model tuned_25d=benchmarks/2026-10-14/tuned_25d
   ```

`evaluate_models.py` 의 `pick_input` 이 **ONNX 입력 차원으로 레이아웃을 자동 판별**하므로 25차원 산출물도 그대로 평가됩니다.

## 기록할 것
- `[Step 0]` 출력의 lightgbm 버전과 런타임 정보(`feature_schema.json` 의 `env` 에도 자동 기록됨)
- `best_iteration` 3종과 지표 → `docs/CHANGELOG.md` `[Unreleased]` 한 줄, 필요 시 `docs/MODEL_PERFORMANCE_REPORT.md` 표 갱신
- 산출물 폴더 `benchmarks/2026-10-14/tuned_25d_colab/` (로컬 `tuned_25d/` 는 대조군으로 **보존**)